Function that will be used to crawl the publication sites and acquire a list of unique publications along with other info

In [2]:
import requests
from bs4 import BeautifulSoup as bsSoup
import time

In [3]:
# scrape the links for all the publications of https://sitescrape.awh.durham.ac.uk/comp42315/publicationfull_year_characteranimation.htm
publicationsIndex = "https://sitescrape.awh.durham.ac.uk/comp42315/publicationfull_year_characteranimation.htm"

def getLinks (url: str) -> list :
    sitePrefix = "https://sitescrape.awh.durham.ac.uk/comp42315/"

    if (not isinstance(url, str)) :
        print("The URL needs to be a string!")
        return []

    # wait a little so we don't overload the server
    time.sleep(2)
    publicationsIndex = requests.get(url, verify = False)

    if (publicationsIndex.status_code != 200) :
        print (f"Error; status code returned: {publicationsIndex.status_code}")
        return []

    publicationsIndexSoup = bsSoup(publicationsIndex.content, "html.parser").body

    if (publicationsIndexSoup == None) :
        print ("Nothin to parse on the site")
        return []

    publicationLinksA = publicationsIndexSoup.find("p", class_ = "TextOption").find_all("a")

    if (publicationLinksA == None) :
        print ("No links found")
        return []
    
    links = [sitePrefix + n.get("href") for n in publicationLinksA]

    if (links[0] == None) :
        print("No links found")
        return []

    return links

publicationLinks = [publicationsIndex] + getLinks(publicationsIndex)

def scrapePublications (urls: list) -> list:
    uniquePublications = set()

    for url in urls :
        content = []

        time.sleep(2)
        publicationSite = requests.get(url, verify = False)
        
        if (publicationSite.status_code != 200) :
            print(f"Failed for link {url}, status code: {publicationSite.status_code}, continuing execution for the remaining links")
            continue

        publicationSiteSoup = bsSoup(publicationSite.content, "html.parser").body

        if (publicationSiteSoup == None) :
            print (f"Found nothing to parse on site {url}, continuing execution for the remaining links")
            continue

        publicationsDiv = publicationSiteSoup.find_all("div", style = "margin-left: var(--size-marginleft)")

        if (len(publicationsDiv) == 0) :
            print (f"Couldn't find publications on site {url}, continuing execution for the remaining links")
            continue

        content = [n.find("div", class_ = "w3-container w3-cell w3-mobile w3-cell-middle").text.strip() for n in publicationsDiv]

    # compare the titles somewhere
    # find first by? and this is a cutoff for the title
    return content 

def scrapeImpactScores (urls: list) -> dict :
    impactScores = {}
    # if a site has been seen, ignore it
    seenSites = set()
    #n = 0
    
    for url in urls :
        #n += 1
        #print(n)
        uniqueScoresOnThePage = []
        # change that 0 to 1 later
        time.sleep(0)
        url = url.replace("year", "impactfactor")
        
        scoreSite = requests.get(url, verify = False)

        if (scoreSite.status_code != 200) :
            print(f"Failed for link {url}, status code: {scoreSite.status_code}, continuing execution for the remaining links")
            continue

        scoreSiteSoup = bsSoup(scoreSite.content, "html.parser").body

        if (scoreSiteSoup == None) :
            print (f"Found nothing to parse on site {url}, continuing execution for the remaining links")
            continue

        scoreSiteSoup = scoreSiteSoup.find("div", id = "divBackground")
        
        if (scoreSiteSoup == None) :
            print (f"Couldn't find scores on the page {url}, continuing execution for the remaining links")
            continue

        scoreSiteSoupP = scoreSiteSoup.find_all("p", class_ = "TextOption")

        if (scoreSiteSoupP == None) :
            print (f"Couldn't find scores on the page {url}, continuing execution for the remaining links")
            continue

        paragraphWithScores = scoreSiteSoupP [2]

        uniqueScoresOnThePage = [n.text for n in paragraphWithScores.find_all("a")]

        if (len(uniqueScoresOnThePage) == 0) :
            print (f"Couldn't find scores on the page {url}, continuing execution for the remaining links")
            continue

        # for each link in unique links find the publication titles that correspond
        for score in uniqueScoresOnThePage :
            currentH2 = scoreSiteSoup.find("h2", id = score)

            if (currentH2 == None) :
                # no publications under this score
                continue

            if (score not in impactScores) :
                impactScores [score] = []

            div = currentH2.findNext()

            publicationsWithThisScore = div.find_all("div", class_ = "w3-cell-row")

            if (publicationsWithThisScore == None) :
                # no publications under this score
                continue

            for publication in publicationsWithThisScore :
                publicationText = publication.find("div", class_ = "w3-container w3-cell w3-mobile w3-cell-middle")

                # I'd need to find the title here
                title = publicationText.text.split("by")[0].strip()

                if (title in seenSites) :
                    continue

                seenSites.add(title)
                impactScores[score].append(publicationText.text)
                #print(title)

    return impactScores
                                                      
#print (publicationLinks)
#print(scrapePublications(publicationLinks))   
scoreDict = scrapeImpactScores(publicationLinks)
print(scoreDict)

c:\Users\Cobra Kai\AppData\Local\Programs\Python\Python314\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'sitescrape.awh.durham.ac.uk'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
c:\Users\Cobra Kai\AppData\Local\Programs\Python\Python314\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'sitescrape.awh.durham.ac.uk'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
C:\Users\Cobra Kai\AppData\Local\Temp\ipykernel_37832\4133510521.py:129: DeprecationWarning: Call to deprecated method findNext. (Replaced by find_next) -- Deprecated since version 4.0.0.
  div = currentH2.findNext()
c:\Users\Cobra Kai\AppData\Local\Programs\Python\Python314\Lib\

{'5.0+': ['\t\r\n\t\tInteraction Patches for Multi-Character Animation by Hubert P. H. Shum, Taku Komura and Shu Takagi in 2010ACM Transactions on Graphics (TOG) - Proceedings of the 2008 ACM SIGGRAPH Asia (SIGGRAPH Asia)\n Webpage\n', '\t\r\n\t\tA Unified Deep Metric Representation for Mesh Saliency Detection and Non-rigid Shape Matching by Brian K. S. Isaac-Medina, Matthew Poyser, Daniel Organisciak, Chris G. Willcocks, Toby P. Breckon and Hubert P. H. Shum in 2020IEEE Transactions on Multimedia (TMM)\n Webpage\n', '\t\r\n\t\tSparse Metric-based Mesh Saliency by John Hartley, Hubert P. H. Shum, Edmond S. L. Ho, He Wang and Subramanian Ramamoorthy in 2021Neurocomputing\n Webpage\n', '\t\r\n\t\tLMZMPM: Local Modified Zernike Moment Per-unit Mass for Robust Human Face Recognition by Luca Crosato, Chongfeng Wei, Edmond S. L. Ho and Hubert P. H. Shum in 2021IEEE Transactions on Information Forensics and Security (TIFS)\n Webpage\n', '\t\r\n\t\tTwo-Stage Human Verification using HandCAPTCH

c:\Users\Cobra Kai\AppData\Local\Programs\Python\Python314\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'sitescrape.awh.durham.ac.uk'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
c:\Users\Cobra Kai\AppData\Local\Programs\Python\Python314\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'sitescrape.awh.durham.ac.uk'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


This information should then be stored and displayed in an appropriate format. Displayed to the user should be: publication title, publication venue, year and author list of every unique publication record on the target website.

In [ ]:
# extract info from score dict - dictionary{title: <>, value [list of the characteristics]}
publications = {}
s = 0

for k in scoreDict :
    for n in scoreDict[k]:
        # 110 (started indexing at 0) unique publications it seems
        remainder = ""
        initialClean = n.translate({ord(i): None for i in '\t\r\n'})
        # remove the "Webpage" at the end
        initialClean_2 = initialClean[:-8]

        # ok, it's important that we only get the first occurance of by
        splitAt = initialClean_2.find("by")
        title = initialClean_2[:splitAt]
        remainder = initialClean_2[splitAt + 3:]

        # works to around there, needs improving this bit
        # as expected there are "in" in the names so that does not work
        splitAt = remainder.find("in")
        authors = remainder[:splitAt]
        year = remainder[splitAt + 3:splitAt + 7]
        remainder = remainder[splitAt + 7:]

        print (s)
        s += 1
        print (title)
        print (remainder)
        print (authors)
        #print (year)
        #print (remainder)
    

0
Interaction Patches for Multi-Character Animation 
ACM Transactions on Graphics (TOG) - Proceedings of the 2008 ACM SIGGRAPH Asia (SIGGRAPH Asia)
Hubert P. H. Shum, Taku Komura and Shu Takagi 
1
A Unified Deep Metric Representation for Mesh Saliency Detection and Non-rigid Shape Matching 
tthew Poyser, Daniel Organisciak, Chris G. Willcocks, Toby P. Breckon and Hubert P. H. Shum in 2020IEEE Transactions on Multimedia (TMM)
Brian K. S. Isaac-Med
2
Sparse Metric-based Mesh Saliency 
Neurocomputing
John Hartley, Hubert P. H. Shum, Edmond S. L. Ho, He Wang and Subramanian Ramamoorthy 
3
LMZMPM: Local Modified Zernike Moment Per-unit Mass for Robust Human Face Recognition 
IEEE Transactions on Information Forensics and Security (TIFS)
Luca Crosato, Chongfeng Wei, Edmond S. L. Ho and Hubert P. H. Shum 
4
Two-Stage Human Verification using HandCAPTCHA and Anti-Spoofed Finger Biometrics with Feature Selection 
by P. Breckon and Hubert P. H. Shum in 2021Expert Systems with Applications (ESWA)

Question 2: <br>
a. Selection of training set. Write code splitting the given dataset into the training and testing sets. The output of the code should be a dataset consisting of the rows of drone.csv that have been chosen. In the text part (part f) of your submission, please justify your choice.

In [66]:
# take a bootstrap sample
import pandas as pd
import random as rd
import matplotlib as mplt
import sklearn as sk

droneData = pd.read_csv("drone.csv")

In [46]:
def takeBootstrapSample (inBagProportion: float, df: pd.DataFrame) -> tuple :
    rowIndicies = []
    for i in range(1,droneData.shape[0] + 1) :
        rowIndicies.append(i)

    trainingSubsetRows = rd.sample(rowIndicies, round(len(rowIndicies) * inBagProportion))
    # converting to sets
    allRowsS = set (rowIndicies)
    trainingRowsS = set (trainingSubsetRows)
    validationSubsetRowsS = allRowsS - trainingRowsS
    # now get the actual rows from the dataframe
    trainingRowsBool = []
    validationRowsBool = []
    for i in rowIndicies :
        if i in trainingRowsS :
            trainingRowsBool.append(True)
        else :
            trainingRowsBool.append(False)

        if i in validationSubsetRowsS :
            validationRowsBool.append(True)
        else :
            validationRowsBool.append(False)

    trainingData = df.iloc[trainingRowsBool]
    validationData = df.iloc[validationRowsBool]

    return (trainingData, validationData)

bootstrapSamp = takeBootstrapSample(0.75, droneData)


b. Feature selection (applied to the training set only). In this task you are requested to write code for testing of two feature selection approaches of your choice. The output of implementation of each approach should be the dataset with three features obtained from the output of part a) by removal of those columns that have not been selected as features.

In [ ]:
trainingDataset = bootstrapSamp[0]

# split the dataframe into the predictors and response
allPredictors = trainingDataset.loc[:, trainingDataset.columns != "warning"]
response = trainingDataset["warning"]

#print (allPredictors.columns)

# convert to numpy array
allPredictorsNp = allPredictors.to_numpy()
responseNp = response.to_numpy()

# calculate the r-regression scores
scores = sk.